In [9]:
import pandas as pd
import numpy as np
import os


# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv("../data/processed/vnl_cleaned.csv")


# ============================================================
# 2. ENSURE DATE IS DATETIME
# ============================================================

df["Y/M/D"] = pd.to_datetime(df["Y/M/D"])






# ============================================================
# 4. CREATE PER-MATCH PERFORMANCE STATISTICS
#
# These are NOT fed directly into the model.
# They are used to calculate historical averages.
# ============================================================



# -------------------------
# Dig Error Rate
# -------------------------

df["DIG_Error_Rate"] = np.where(
    df["DIG_Digs"] > 0,
    df["DIG_Errors"] / df["DIG_Digs"],
    np.nan
)


# -------------------------
# Set Error Rate
# -------------------------

df["SET_Error_Rate"] = np.where(
    df["SET_Attempts"] > 0,
    df["SET_Errors"] / df["SET_Attempts"],
    np.nan
)


# ============================================================
# 5. DEFINE PERFORMANCE FEATURES
# ============================================================

performance_features = [
    "ATTACK_Efficiency",
    "SERVE_Efficiency",
    "RECEPTION_Efficiency",
    "BLOCK_Efficiency",
    "DIG_Error_Rate",
    "SET_Error_Rate"
]


# ============================================================
# 6. CREATE HISTORICAL FEATURES
#
# IMPORTANT:
#
# Features for matches on Date X only use data from
# matches BEFORE Date X.
#
# This prevents same-day leakage.
# ============================================================


def calculate_historical_features(team_data):
    """
    Calculates historical features for one team.

    Statistics from a match are NEVER used to predict
    that same match.
    """

    team_data = team_data.sort_values(
        ["Y/M/D", "Match_ID"]
    ).copy()

    # Store all completed historical matches
    history = []

    results = []

    # Process one date at a time
    for date, date_matches in team_data.groupby(
        "Y/M/D",
        sort=True
    ):

        # Calculate features for every match on this date
        # BEFORE adding this date's matches to history

        if len(history) == 0:

            historical_values = {
                "Prior_Win_Rate": np.nan,
                "Recent_Form": np.nan
            }

            for feature in performance_features:
                historical_values[
                    f"Prior_{feature}"
                ] = np.nan

        else:

            history_df = pd.DataFrame(history)

            historical_values = {

                # Overall historical win rate
                "Prior_Win_Rate":
                    history_df["Won"].mean(),


                # Win rate from last 10 matches
                "Recent_Form":
                    history_df["Won"]
                    .tail(40)
                    .mean()
            }

            # Last 5-match rolling averages
            for feature in performance_features:

                historical_values[
                    f"Prior_{feature}"
                ] = (
                    history_df[feature]
                    .tail(40)
                    .mean()
                )

        # Apply the same historical values
        # to every match on this date
        for _, row in date_matches.iterrows():

            row_dict = row.to_dict()

            row_dict.update(
                historical_values
            )

            results.append(
                row_dict
            )

        # Only AFTER features have been created,
        # add today's matches to history
        for _, row in date_matches.iterrows():

            history.append(
                row[
                    ["Won"]
                    + performance_features
                ].to_dict()
            )

    return pd.DataFrame(results)


# ============================================================
# 7. APPLY HISTORICAL FEATURE ENGINEERING
#    TO EVERY TEAM
# ============================================================

historical_dfs = []

for team, team_data in df.groupby("Team"):

    processed_team = calculate_historical_features(
        team_data
    )

    historical_dfs.append(
        processed_team
    )


df_features = pd.concat(
    historical_dfs,
    ignore_index=True
)


# ============================================================
# 8. SORT AGAIN
# ============================================================

df_features = df_features.sort_values(
    ["Y/M/D", "Match_ID", "Team"]
).reset_index(drop=True)


# ============================================================
# 9. DEFINE HISTORICAL FEATURES
# ============================================================

prior_features = [
    "Prior_Win_Rate",
    "Recent_Form",
    "Prior_ATTACK_Efficiency",
    "Prior_SERVE_Efficiency",
    "Prior_RECEPTION_Efficiency",
    "Prior_BLOCK_Efficiency",
    "Prior_DIG_Error_Rate",
    "Prior_SET_Error_Rate"
]


# ============================================================
# 10. CREATE TEAM DATAFRAME
#
# This represents the match from the perspective
# of the current Team.
# ============================================================

team_df = df_features[
    [
        "Match_ID",
        "Y/M/D",
        "Team",
        "VS_Team",
        "Won"
    ]
    + prior_features
].copy()


# ============================================================
# 11. CREATE OPPONENT DATAFRAME
#
# We take the opponent's historical statistics
# and merge them onto the current team's row.
# ============================================================

opponent_df = df_features[
    ["Match_ID", "Team"]
    + prior_features
].copy()


opponent_df = opponent_df.rename(
    columns={
        "Team": "VS_Team",

        **{
            feature:
            f"VS_{feature}"

            for feature
            in prior_features
        }
    }
)


# ============================================================
# 12. MERGE TEAM + OPPONENT FEATURES
# ============================================================

match_df = team_df.merge(
    opponent_df,

    on=[
        "Match_ID",
        "VS_Team"
    ],

    how="left"
)


# ============================================================
# 13. CREATE DIFFERENCE FEATURES
#
# Team statistic - Opponent statistic
# ============================================================

for feature in prior_features:

    match_df[
        f"{feature}_Diff"
    ] = (

        match_df[feature]

        -

        match_df[
            f"VS_{feature}"
        ]
    )


# ============================================================
# 14. SELECT FEATURES FOR THE MODEL
# ============================================================

feature_columns = [

    "Prior_Win_Rate_Diff",

    "Recent_Form_Diff",

    "Prior_ATTACK_Efficiency_Diff",

    "Prior_SERVE_Efficiency_Diff",

    "Prior_RECEPTION_Efficiency_Diff",

    "Prior_BLOCK_Efficiency_Diff",

    "Prior_DIG_Error_Rate_Diff",

    "Prior_SET_Error_Rate_Diff"
]


# ============================================================
# 15. REMOVE ROWS WITHOUT ENOUGH HISTORY
#
# For example, a team's first ever match will
# not have previous statistics.
# ============================================================

match_df = match_df.dropna(
    subset=feature_columns
)


# ============================================================
# 16. CREATE X AND y
# ============================================================

X = match_df[
    feature_columns
]

y = match_df[
    "Won"
]


# ============================================================
# 17. SAVE FEATURE DATASET
# ============================================================

os.makedirs("../data/features", exist_ok=True)

match_df.to_csv(
    "../data/features/vnl_features.csv",
    index=False
)


# ============================================================
# 18. CHECK RESULTS
# ============================================================

print("Feature engineering complete!")

print()

print("Feature dataset shape:")
print(match_df.shape)

print()

print("Features:")
print(feature_columns)

print()

print("X shape:")
print(X.shape)

print()

print("y shape:")
print(y.shape)

print()

print("First few rows:")
print(
    match_df[
        [
            "Y/M/D",
            "Match_ID",
            "Team",
            "VS_Team",
            "Won"
        ]
        + feature_columns
    ].head()
)

Feature engineering complete!

Feature dataset shape:
(1070, 29)

Features:
['Prior_Win_Rate_Diff', 'Recent_Form_Diff', 'Prior_ATTACK_Efficiency_Diff', 'Prior_SERVE_Efficiency_Diff', 'Prior_RECEPTION_Efficiency_Diff', 'Prior_BLOCK_Efficiency_Diff', 'Prior_DIG_Error_Rate_Diff', 'Prior_SET_Error_Rate_Diff']

X shape:
(1070, 8)

y shape:
(1070,)

First few rows:
        Y/M/D                         Match_ID       Team        VS_Team  Won  \
16 2021-05-29      2021-05-29_Argentina_Canada  Argentina         Canada    0   
17 2021-05-29      2021-05-29_Argentina_Canada     Canada      Argentina    1   
18 2021-05-29    2021-05-29_Australia_Bulgaria  Australia       Bulgaria    0   
19 2021-05-29    2021-05-29_Australia_Bulgaria   Bulgaria      Australia    1   
20 2021-05-29  2021-05-29_Brazil_United States     Brazil  United States    1   

    Prior_Win_Rate_Diff  Recent_Form_Diff  Prior_ATTACK_Efficiency_Diff  \
16                  0.0               0.0                      0.514161   
1

In [7]:
team_df.to_clipboard()

In [8]:
historical_dfs[0].to_clipboard()

In [9]:
import pandas as pd
import os



df = pd.read_csv("/Users/zu/Desktop/projects/vballPredictor/data/features/vnl_features.csv")
df.to_clipboard()